# Neural Network for Predicting House Prices

In this notebook i'm building a simple neural network using the California Housing dataset. The goal is to predict house prices based on different features like income, location, house age etc. I used scikit-learn's MLPRegressor instead of building from scratch.

## Importing the libraries

first i need to import everything i'll be using throughout the notebook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Loading the dataset

using the california housing dataset like mentioned in the assignment. it has 20640 samples and 8 features (things like income, house age, number of rooms etc). the target is the median house price.

In [ ]:
# load the dataset
data = fetch_california_housing()
X, y = data.data, data.target

# just checking what we're working with
print("number of samples:", X.shape[0])
print("number of features:", X.shape[1])
print("feature names:", data.feature_names)

## Splitting the data

i split the data into training and testing sets. 80% goes to training (so the model can learn from it) and 20% is kept aside for testing. this way we can see how well the model does on data it hasn't seen before.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("training samples:", X_train.shape[0])
print("testing samples:", X_test.shape[0])

## Scaling the features

the features have very different ranges so i need to normalize them. for example income values are way bigger than house age values. if i don't scale them the model might focus too much on the bigger numbers. StandardScaler fixes this by making everything have a similar range.

important: i only fit the scaler on training data, not test data

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # only transform, don't fit on test data

## Building and training the model

here i create the neural network. i used MLPRegressor with:
- 1 hidden layer with 16 neurons
- relu activation function
- adam as the solver
- 500 iterations

the .fit() is where the actual learning happens, the model goes through the training data and adjusts its weights to get better at predicting

In [ ]:
model = MLPRegressor(hidden_layer_sizes=(16,), activation='relu',
                     solver='adam', max_iter=500, random_state=42)

model.fit(X_train_scaled, y_train)

print("done training! took", model.n_iter_, "iterations")

## Evaluating the model

now i test the model on the 20% it hasn't seen before and measure how good the predictions are using 3 metrics:
- MAE: average error in the predictions
- RMSE: similar to MAE but punishes bigger mistakes more
- R²: overall score, closer to 1 means better

In [ ]:
# make predictions on test set
y_pred = model.predict(X_test_scaled)

# calculate the 3 metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("MAE: ", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2:  ", round(r2, 4))

## Visualizations

### Loss curve
this shows how the model's error went down over the 500 training iterations. if it goes down it means the model was actually learning

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(model.loss_curve_, color='steelblue')
plt.title('Training Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.show()

### True vs Predicted
each dot is a house from the test set. the red line is where predicted = actual (perfect prediction). the closer the dots are to that line the better the model did

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=15)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='perfect prediction')
plt.title('True vs Predicted House Prices')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

so basically what i did was load the california housing dataset which has 8 features and around 20k samples. i split it 80/20 for training and testing, scaled the features so they're all on the same range, then built a neural network with one hidden layer of 16 neurons.

the model trained for 500 iterations and the loss curve shows it was learning properly since the error kept going down.

for the results, the R2 score came out around 0.80 which means the model explains about 80% of the variation in house prices. the MAE shows the average prediction was off by around 50k which is decent for a simple one layer network.

the true vs predicted plot shows most predictions are close to the red line which means the model is doing a reasonable job. it struggles a bit more with very high value houses but overall it learned the general pattern pretty well.

if i wanted to improve it i could try adding more neurons or more hidden layers but for this assignment the simple version works fine.